### Dependencies

In [134]:
import itertools

import torch
import torch.nn.functional as F

### Dataset

In [135]:
with open("data/names.txt", "r") as f:
    words = f.read().splitlines()

In [136]:
chars = sorted(set("".join(words)))
stoi = {ch: i for i, ch in enumerate(chars)}
stoi["."] = len(stoi)
itos = {ch: i for i, ch in stoi.items()}

print(stoi)
print(itos)

{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, '.': 26}
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z', 26: '.'}


In [137]:
xs, ys = [], []

for w in words:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2 in itertools.pairwise(chs):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [138]:
xenc = F.one_hot(xs, num_classes=len(stoi)).to(torch.float32)
print(xenc.shape)
print(xenc.dtype)

torch.Size([228146, 27])
torch.float32


### Neural Network

In [139]:
g = torch.Generator().manual_seed(2871305)
W = torch.randn(len(stoi), len(stoi), generator=g, requires_grad=True)

### Training

In [140]:
learning_rate = 100
regularization = 0.01

for i in range(100):
    # forward pass
    xenc = F.one_hot(xs, num_classes=len(stoi)).float()
    logits = xenc @ W

    # softmax to get probabilities
    counts = logits.exp()
    probs = counts / counts.sum(dim=1, keepdim=True)

    loss = (
        -probs[torch.arange(ys.shape[0]), ys].log().mean()
        + regularization * (W**2).mean()
    )
    print(loss.item())

    # backward pass
    W.grad = None
    loss.backward()

    # update weights
    W.data += -learning_rate * W.grad

3.876189947128296
3.1501338481903076
2.904366970062256
2.797813892364502
2.7280962467193604
2.7051355838775635
2.6594295501708984
2.643723249435425
2.616987466812134
2.6150805950164795
2.5892672538757324
2.588331699371338
2.5732181072235107
2.5800271034240723
2.5598275661468506
2.5634853839874268
2.552734136581421
2.563239097595215
2.5446908473968506
2.5503721237182617
2.5418965816497803
2.5549652576446533
2.536264419555664
2.5430526733398438
2.5355122089385986
2.550447702407837
2.530454158782959
2.537490129470825
2.5306248664855957
2.5468246936798096
2.525693893432617
2.5324668884277344
2.5266916751861572
2.543881893157959
2.521902322769165
2.5282235145568848
2.523620843887329
2.541646718978882
2.5189411640167236
2.5247743129730225
2.521202325820923
2.5399162769317627
2.51662015914917
2.52201771736145
2.519256591796875
2.5385050773620605
2.514784574508667
2.519848108291626
2.5176656246185303
2.5373027324676514
2.5133166313171387
2.5181570053100586
2.5163536071777344
2.536257028579712


### Sample from Neural Network

In [141]:
g = torch.Generator().manual_seed(2871305)


for i in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=len(stoi)).float()
        logits = xenc @ W
        counts = logits.exp()
        probs = counts / counts.sum(dim=1, keepdim=True)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        out.append(itos[ix])
        if itos[ix] == ".":
            break
    print("".join(out))

mukleswa.
inp.
.
hon.
yen.
